# Notebook 01 — Systematic Exploration (Recalls)

**Purpose:** NHTSA Recalls API hypotheses verification after initial exploration.

**Hypotheses to verify:**
1. Hierarchical structure: one `NHTSACampaignNumber` → multiple `NHTSAActionNumber` (per affected model and production year)
2. Date format DD/MM/YYYY (observed "28/05/2020" for example, will be confirmed on larger sample)
3. Null distribution across fields (small initial exploration sample showed none, need more records)
4. Hard cap on records per query (for example - Tesla S (2022) returned 20 — try higher)

**Sample plan:** ~200-500 records across diverse manufacturers and years.

**Hypothesis context:** see Recalls section in `data_profiling.md` for full context.

In [61]:
import requests
import pandas as pd
from pprint import pprint

BASE_URL = "https://api.nhtsa.gov/recalls/recallsByVehicle"

def fetch_recalls(make, model, year):
    params = {"make": make, "model": model, "modelYear": year}
    response = requests.get(BASE_URL, params = params)
    response.raise_for_status()
    return response.json().get("results", [])

def fetch_recall_count(make, model, year):
    params = {"make": make, "model": model, "modelYear": year}
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json().get("Count", 0)

In [ ]:
# Hypothesis 1 test: one campaign should appear across multiple model queries
# Models likely affected by the same VW campaign (from initial exploration  observation):
# "...2019 Jetta GLI, Golf Alltrack, and Golf GTI vehicles"

models_to_check = [
    ("volkswagen", "jetta", 2019),
    ("volkswagen", "golf alltrack", 2019),
    ("volkswagen", "gti", 2019),
]

# Collect campaign numbers for each query
results_per_query = {}

for make, model, year in models_to_check:
    recalls = fetch_recalls(make, model, year)
    campaign_numbers = [r.get("NHTSACampaignNumber") for r in recalls]
    results_per_query[f"{make} {model} {year}"] = campaign_numbers
    
    print(f"{make} {model} {year}: {len(recalls)} recalls")
    print(f"  Campaigns: {campaign_numbers}")
    print()

volkswagen jetta 2019: 11 recalls
  Campaigns: ['19V679000', '18V671000', '18V904000', '18V824000', '19V110000', '19V188000', '19V879000', '22V815000', '23V604000', '23V619000', '24V110000']

volkswagen golf alltrack 2019: 3 recalls
  Campaigns: ['19V879000', '22V815000', '23V604000']

volkswagen gti 2019: 4 recalls
  Campaigns: ['19V615000', '22V815000', '23V604000', '24V110000']



In [ ]:
# Example of shared campaign across multiple models
shared_campaign = '22V815000'

print(f"All records inspection for NHTSACampaignNumber: {shared_campaign}")

for make, model, year in models_to_check:
    recalls = fetch_recalls(make, model, year)
    matching = [r for r in recalls if r.get("NHTSACampaignNumber") == shared_campaign]
    
    if matching:
        record = matching[0]
        print(f"\nFrom query: {make} {model} {year}")
        print(f"  NHTSAActionNumber:  {record.get('NHTSAActionNumber')}")
        print(f"  Manufacturer:        {record.get('Manufacturer')}")
        print(f"  Make returned:       {record.get('Make')}")
        print(f"  Model returned:      {record.get('Model')}")
        print(f"  ModelYear:           {record.get('ModelYear')}")
        print(f"  Component:           {record.get('Component')}")
        print(f"  ReportReceivedDate:  {record.get('ReportReceivedDate')}")
        print(f"  Summary (first 100): {record.get('Summary', '')[:100]}...")

All records inspection for NHTSACampaignNumber: 22V815000

From query: volkswagen jetta 2019
  NHTSAActionNumber:  None
  Manufacturer:        Volkswagen Group of America, Inc.
  Make returned:       VOLKSWAGEN
  Model returned:      JETTA
  ModelYear:           2019
  Component:           TIRES:PRESSURE MONITORING AND REGULATING SYSTEMS
  ReportReceivedDate:  31/10/2022
  Summary (first 100): Volkswagen Group of America, Inc. (Volkswagen) is recalling certain 2019 Volkswagen Tiguan LWB, Golf...

From query: volkswagen golf alltrack 2019
  NHTSAActionNumber:  None
  Manufacturer:        Volkswagen Group of America, Inc.
  Make returned:       VOLKSWAGEN
  Model returned:      GOLF ALLTRACK
  ModelYear:           2019
  Component:           TIRES:PRESSURE MONITORING AND REGULATING SYSTEMS
  ReportReceivedDate:  31/10/2022
  Summary (first 100): Volkswagen Group of America, Inc. (Volkswagen) is recalling certain 2019 Volkswagen Tiguan LWB, Golf...

From query: volkswagen gti 2019
  NHTSA

### Hypothesis 1 — Verification Result

**Status:** Modified

**Findings:**
- Confirmed: one `NHTSACampaignNumber` can affect multiple models, appearing as separate API records (one per model) with identical campaign-level fields.
- Rejected: `NHTSAActionNumber` is **not** an action-per-model identifier. It is an **optional** field and likely references a NHTSA Investigation (`RQ`-prefixed) that led to the recall.

**Architectural implication:** Single fact table (grain: campaign × affected_model) with dimensions `dim_campaign`, `dim_model`, `dim_investigation` (one row per `NHTSAActionNumber`, connected with `fact_recalls` through nullable FK — there are recalls without investigation link).

In [ ]:
# Hypothesis 2 test: dates present in `ReportReceivedDate` should have European representation: DD/MM/YYYY.
# Approach: Collect dates from diverse manufactures and/or years. parse first and middle components, check if middle component stays in 1-12 range (months) or extends to 1-31 (days).
sample_queries = [
    ("toyota", "camry", 2020),
    ("toyota", "corolla", 2021),
    ("toyota", "rav4", 2021),
    ("ford", "f-150", 2018),
    ("ford", "explorer", 2022),
    ("ford", "escape", 2021),
    ("honda", "civic", 2017),
    ("honda", "cr-v", 2023),
    ("honda", "accord", 2022),
    ("chevrolet", "equinox", 2012),
    ("chevrolet", "malibu", 2010),
    ("nissan", "altima", 2020),
    ("hyundai", "tucson", 2023),
    ("tesla", "model 3", 2022),
    ("tesla", "model y", 2024),
    ("bmw", "m3", 2021),
    ("volkswagen", "jetta", 2019),
    ("hyundai", "elantra", 2016),
    ("kia", "optima", 2019),
    ("kia", "sportage", 2022),
    ("subaru", "outback", 2021),
    ("jeep", "grand cherokee", 2020),
    ("jeep", "wrangler", 2017),
    ("ram", "1500", 2022),
    ("gmc", "sierra 1500", 2021),
    ("toyota", "camry", 2015),
    ("toyota", "corolla", 2016),
    ("toyota", "rav4", 2018),
    ("ford", "f-150", 2019),
    ("ford", "explorer", 2016),
    ("ford", "escape", 2014),
    ("honda", "civic", 2022),
    ("honda", "cr-v", 2022),
    ("honda", "accord", 2016),
    ("chevrolet", "equinox", 2015),
    ("chevrolet", "malibu", 2017),
    ("nissan", "altima", 2021),
    ("hyundai", "tucson", 2025),
    ("tesla", "model 3", 2021),
    ("tesla", "model y", 2025),
    ("volkswagen", "jetta", 2017),
    ("hyundai", "elantra", 2014),
    ("kia", "optima", 2018),
    ("kia", "sportage", 2025),
    ("subaru", "outback", 2022),
    ("jeep", "grand cherokee", 2013),
    ("jeep", "wrangler", 2020),
    ("ram", "1500", 2017),
    ("gmc", "sierra 1500", 2020)
]

all_records = []
for make, model, year in sample_queries:
    recalls = fetch_recalls(make, model, year)
    all_records.extend(recalls)

print(f"Total records collected: {len(all_records)}")
print(f"Total  (make x model x year) combinations examined: {len(sample_queries)}")

Total records collected: 323
Total  model x production year examined: 49


In [19]:
# Parse dates and check format consistency
date_analysis = []

for record in all_records:
    campaign = record.get("NHTSACampaignNumber", "")
    date_str = record.get("ReportReceivedDate", "")
    
    if not date_str or "/" not in date_str:
        continue  # skip if no date
    
    parts = date_str.split("/")
    if len(parts) != 3:
        continue  # skip if not formatted correctly
    
    first, middle, last = parts
    
    # Year encoded in campaign prefix (first 2 chars)
    campaign_year_prefix = campaign[:2] if campaign else None
    
    date_analysis.append({
        "campaign": campaign,
        "campaign_year_prefix": campaign_year_prefix,
        "date_raw": date_str,
        "first": int(first),
        "middle": int(middle),
        "last": int(last),
    })

# Convert to DataFrame for analysis
df_dates = pd.DataFrame(date_analysis)
print(f"Parsed records: {len(df_dates)}")
print()
print("First component range:", df_dates["first"].min(), "to", df_dates["first"].max())
print("Middle component range:", df_dates["middle"].min(), "to", df_dates["middle"].max())

Parsed records: 323

First component range: 1 to 31
Middle component range: 1 to 12


In [20]:
# Cross-validation: campaign year prefix should match year from date (last component)
df_dates["last_year_2digit"] = df_dates["last"] % 100  # last 2 digits of year
df_dates["year_match"] = df_dates["campaign_year_prefix"] == df_dates["last_year_2digit"].astype(str).str.zfill(2)

match_count = df_dates["year_match"].sum()
total_count = len(df_dates)

print(f"Records where campaign prefix matches date year: {match_count} / {total_count}")
print(f"Match rate: {match_count / total_count:.1%}")
print()

# Show mismatches if the are present
mismatches = df_dates[~df_dates["year_match"]]
if len(mismatches) > 0:
    print(f"Mismatches found ({len(mismatches)} records):")
    print(mismatches[["campaign", "campaign_year_prefix", "date_raw", "last"]].head(10))
else:
    print("No mismatches — perfect cross-validation.")

Records where campaign prefix matches date year: 322 / 323
Match rate: 99.7%

Mismatches found (1 records):
      campaign campaign_year_prefix    date_raw  last
192  20V675000                   20  12/03/2021  2021


In [26]:
# Inspect the mismatching record
target_campaign = "20V675000"

# Find which query returned it
for make, model, year in sample_queries:
    recalls = fetch_recalls(make, model, year)
    for r in recalls:
        if r.get("NHTSACampaignNumber") == target_campaign:
            print(f"Found in query: {make} {model} {year}")
            print(f"ReportReceivedDate: {r.get('ReportReceivedDate')}")
            print()
            print("Summary:")
            print(r.get("Summary"))
            print()
            print("Remedy:")
            print(r.get("Remedy"))
            print()
            print("Notes:")
            print(r.get("Notes"))
            break
    else:
        continue
    break

Found in query: ford explorer 2016
ReportReceivedDate: 12/03/2021

Summary:
Ford Motor Company (Ford) is recalling certain 2013-2017 Explorer vehicles originally sold, or currently registered in Connecticut, Delaware, the District of Columbia, Illinois, Indiana, Iowa, Kentucky, Maine, Maryland, Massachusetts, Michigan, Minnesota, Missouri, New Hampshire, New Jersey, New York, Ohio, Pennsylvania, Rhode Island, Vermont, Virginia, West Virginia, and Wisconsin that were previously repaired under a prior recall numbers 16V-245 or 19V-435.  The outboard section of a rear suspension toe link may fracture.

Remedy:
Ford will notify owners, and dealers will inspect the cross-axis ball joint (CABJ) knuckle attached to the rear suspension toe link and replace it as necessary, free of charge.  The recall began November 27, 2020.  Owners may contact Ford customer service at 1-866-436-7332.  Ford's number for this recall is 20S62.

Notes:
Owners may also contact the National Highway Traffic Safety A

### Hypothesis 2 — Verification Result

**Status:** Confirmed

**Findings:**
- Confirmed: `ReportReceivedDate` represent European date formatting DD/MM/YYYY based on the observation of 323-element sample for 49 combination: (make × model × year). First component of the date represents the values from the range **1 to 31**, and the middle one **1 to 12** indicating respectively day and month. 
The year prefix of `NHTSACampaignNumber` matches the year of `ReportReceivedDate` in 99.7% in records, suggesting both fields reference the same underlying campaign-opening date.
- **Edge case (1 / 323 records):** `NHTSACampaignNumber: 20V675000` with `ReportReceivedDate: 12/03/2021` — likely a recall amendment where date was updated when scope expanded, while campaign number was preserved.

**Architectural implication:** 
- Silver layer: parse `ReportReceivedDate` using `pd.to_datetime(..., format="%d/%m/%Y")`
- Gold layer: `fact_recall` will link to `dim_date` through a `report_received_date_key` (surrogate integer key, e.g. `20230930` for YYYYMMDD)
- Derived field `is_recall_amendment` in silver, based on text patterns in `Remedy` / `Summary` ("expands", "amendment", "previously recalled"); date-prefix mismatch may serve as a secondary signal but is not the primary classifier
- dbt test: cross-field check `campaign_year_prefix == date_year`, **severity: warn** (expected ~0.3% legitimate mismatches due to amendments)

In [ ]:
# Hypothesis 3 test: missing values presence
# Sample: 323 records from 49 (make × model × year) combinations

# Convert all_records list to DataFrame 
# Placehoders considered as missing values - they are also counted
df_records = pd.DataFrame(all_records)
df_records.replace(["", "N/A", "Unknown", "-", "null"], None, inplace = True)

# Missing values summary table: count and percentage sorted descending
missing_summary = pd.DataFrame({
    "missing_count": df_records.isna().sum(),
    "missing_pct": df_records.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

print(f"Total records: {len(df_records)}")
print(f"Total fields: {len(df_records.columns)}\n")
missing_summary

Total records: 323
Total fields: 15



,missing_count,missing_pct
NHTSAActionNumber,298,92.260062
Notes,40,12.383901
parkIt,5,1.547988
overTheAirUpdate,5,1.547988
parkOutSide,5,1.547988
NHTSACampaignNumber,0,0.000000
Manufacturer,0,0.000000
Component,0,0.000000
ReportReceivedDate,0,0.000000
Consequence,0,0.000000


In [48]:
# Check the presence of  misssing values for severity flags
severity_cols = ["parkIt", "parkOutSide", "overTheAirUpdate"]
all_three_null = df_records[severity_cols].isna().all(axis=1).sum()
any_null = df_records[severity_cols].isna().any(axis=1).sum()

print(f"Records with ALL three severity flags null: {all_three_null}")
print(f"Records with ANY severity flag null: {any_null}")

Records with ALL three severity flags null: 5
Records with ANY severity flag null: 5


### Hypothesis 3 — Verification Result

**Status:** Modified

**Findings:**
- Confirmed: Missing values are present in the 323-record sample at varying rates per field. 
`NHTSAActionNumber` is missing in ~92.26% of records — confirming earlier observation that this field is optional. Its semantic role (likely a reference to prior NHTSA Investigation, based on observed RQ-prefixed values) requires NHTSA documentation verification. 
`Notes` is missing in ~12.38% which indicates a non-mandatory additional information.
Severity flags: `parkIt`, `parkOutSide`, and `overTheAirUpdate` show identical null rates with missing values **in the same records**.

**Critical fields:** No missing values observed for `NHTSACampaignNumber`, `Manufacturer`, `Make`, `Model`, `ModelYear`, `ReportReceivedDate`, `Component`, `Summary`, `Consequence`, `Remedy`. These align with the **hard constraint** classification proposed in earlier profiling.

**Architectural implication:** 
- Silver layer must enforce `not_null` tests for fields confirmed as 100% complete (campaign key, manufacturer, model, dates, narrative fields).
- `NHTSAActionNumber` to be loaded as nullable; only non-null values are used to populate `dim_investigation` (per earlier model decision).
- Records with null `parkIt`, `parkOutSide`, and `overTheAirUpdate` need further investigation — if confirmed coincident, may indicate older recall records pre-dating the introduction of these severity classifications by NHTSA. Worth checking the date distribution of records with missing severity flags.
- All severity-flag null records to be flagged with a derived column `has_complete_severity_classification` (boolean) for downstream filtering of severity-weighted analyses.

In [62]:
# Hypothesis 4: Hard cap on records per query
max_count = 0
max_vehicle = None

edge_case_queries = [
("honda", "accord", 2003),
("honda", "accord", 2004),
("honda", "accord", 2005),
("honda", "accord", 2006),
("honda", "accord", 2007),
("honda", "civic", 2003),
("honda", "civic", 2004),
("honda", "civic", 2005),
("honda", "civic", 2006),
("honda", "civic", 2007),
("toyota", "corolla", 2003),
("toyota", "corolla", 2004),
("toyota", "corolla", 2005),
("toyota", "corolla", 2006),
("toyota", "corolla", 2007),
("toyota", "corolla", 2008),
("bmw", "m3", 2000),
("bmw", "m3", 2001),
("bmw", "m3", 2002),
("bmw", "m3", 2003),
("bmw", "m3", 2004),
("bmw", "m3", 2005),
("bmw", "m3", 2006),
]

for make, model, year in edge_case_queries:
    count = fetch_recall_count(make, model, year)

    if count > max_count:
        max_count = count
        max_vehicle = (make, model, year)

print(max_count)
print(max_vehicle)

24
('honda', 'accord', 2003)


In [54]:
recalls_accord = fetch_recalls("honda", "accord", 2003)

components = [r.get("Component") for r in recalls_accord]
print(f"Total recalls: {len(recalls_accord)}\n")

# Count occurrences of each component
from collections import Counter
component_counts = Counter(components)
for component, count in component_counts.most_common():
    print(f"  {count}x — {component}")

Total recalls: 24

  6x — EXTERIOR LIGHTING:HEADLIGHTS
  4x — AIR BAGS:FRONTAL:PASSENGER SIDE:INFLATOR MODULE
  4x — EXTERIOR LIGHTING
  3x — AIR BAGS:FRONTAL:DRIVER SIDE:INFLATOR MODULE
  2x — AIR BAGS
  1x — VEHICLE SPEED CONTROL:ACCELERATOR PEDAL
  1x — STEERING:HYDRAULIC POWER ASSIST:HOSE, PIPING, AND CONNECTIONS
  1x — ELECTRICAL SYSTEM:IGNITION
  1x — VISIBILITY:WINDSHIELD WIPER/WASHER:MOTOR
  1x — POWER TRAIN:AUTOMATIC TRANSMISSION


### Hypothesis 4 — Verification Result

**Status:** Confirmed 

**Findings:**

- Empirically tested high-recall edge cases (Honda Accord 2003-2007, Honda Civic 2003-2007, Toyota Corolla 2003-2008, BMW M3 2000-2006, etc.)
- Maximal recall number observed: **24 records** (Honda Accord 2003)
- No round-number (e.g. 20, 50, 100) response sizes that would indicate a cap
- Drill-down on Honda Accord 2003: from 24 recalls, 9 relate to airbags (multiple Takata campaigns and amendments), remaining 15 cover other components. This confirms the count represents **administrative campaigns**, not defect instances or affected vehicles.

**Architectural implication:**

- `recallsByVehicle` endpoint at (make, model, year) granularity returns complete data in a single request